# Model 04 — Gradient Boosting

## BRFSS 2025 Diabetes Classification

**Purpose:** independently execute and freeze the fourth classifier in the benchmark without rerunning Models 01–03.

### Literature provenance

**BRFSS application:** Muhammad et al. (2025), *Journal of Primary Care & Community Health*, DOI `10.1177/21501319251400546`, evaluated Gradient Boosting as one of seven classifiers on BRFSS 2023 using stratified 5-fold CV.

**Architecture source:** Friedman (2001), *Greedy Function Approximation: A Gradient Boosting Machine*, *The Annals of Statistics*, DOI `10.1214/aos/1013203451`.

Gradient boosting builds an additive classifier sequentially. Each new shallow regression tree is fit to the negative gradient of the current loss, so later stages correct errors left by earlier stages.

This notebook uses the frozen Part II cohort, features, preprocessing, split, and evaluation protocol. It does **not** retune prior models.


## 1. Frozen Study Protocol

The following are inherited unchanged from Part II:

- target: `DIABETE4=1` versus `DIABETE4=3`;
- 20 frozen predictors;
- 80/20 stratified split with `random_state=42`;
- categorical missing values represented explicitly and one-hot encoded;
- continuous missing values median-imputed;
- continuous predictors standardized;
- Stratified 5-Fold CV on training data only;
- no SMOTE or class resampling in the primary benchmark;
- untouched test set evaluated once after model specification is frozen.

The test set is never used to select Gradient Boosting hyperparameters.


In [ ]:
from pathlib import Path
from math import sqrt
import json
import platform
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "DIABETE4"

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

xpt_candidates = glob.glob("/kaggle/input/**/LLCP2025.XPT", recursive=True)
if len(xpt_candidates) != 1:
    raise RuntimeError(f"Expected exactly one LLCP2025.XPT input, found: {xpt_candidates}")
xpt_path = Path(xpt_candidates[0])

print("Input:", xpt_path)
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)

    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)

    return out[PRIMARY_FEATURES]

target_mask = df[TARGET].isin([1.0, 3.0])
X = recode_brfss_2025(df.loc[target_mask, PRIMARY_FEATURES].copy())
y = (df.loc[target_mask, TARGET] == 1.0).astype("int8").rename("diabetes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

assert len(X) == 342_539
assert len(X_train) == 274_031
assert len(X_test) == 68_508
assert int(y.sum()) == 51_827

print("Cohort:", len(X))
print("Train:", len(X_train), "Test:", len(X_test))
print("Train diabetes %:", y_train.mean() * 100)
print("Test diabetes %:", y_test.mean() * 100)


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

specificity_scorer = make_scorer(recall_score, pos_label=0)
precision_scorer = make_scorer(precision_score, zero_division=0)
scoring = {
    "accuracy": "accuracy",
    "precision": precision_scorer,
    "recall": "recall",
    "specificity": specificity_scorer,
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

print("Frozen preprocessing and CV protocol ready.")


## 2. Model Architecture

The fixed baseline is:

```python
GradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.1,
    n_estimators=100,
    subsample=1.0,
    criterion="friedman_mse",
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    max_features=None,
    random_state=42,
)
```

### Provenance classification

**Literature/architecture-supported**
- Gradient Boosting model family;
- sequential additive trees minimizing classification loss;
- use on BRFSS diabetes;
- 5-fold CV and held-out evaluation.

**Study-specific fixed choices**
- 100 stages;
- learning rate 0.1;
- depth-3 weak learners;
- no stochastic subsampling;
- no class weighting/resampling;
- threshold 0.5.

These values are fixed before test evaluation and are not claimed to reproduce the tuned P03 configuration.


In [ ]:
MODEL_04_DIR = Path("/kaggle/working/model_04_gradient_boosting")
MODEL_04_DIR.mkdir(parents=True, exist_ok=True)

gradient_boosting = GradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.1,
    n_estimators=100,
    subsample=1.0,
    criterion="friedman_mse",
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    max_features=None,
    random_state=RANDOM_STATE,
)

gb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", gradient_boosting),
])

config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "GradientBoostingClassifier",
    "architecture": "forward stage-wise additive regression-tree ensemble",
    "loss": "log_loss",
    "learning_rate": 0.1,
    "n_estimators": 100,
    "subsample": 1.0,
    "criterion": "friedman_mse",
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_depth": 3,
    "max_features": None,
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_04_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


## 3. Five-Fold Cross-Validation

The full preprocessing + Gradient Boosting pipeline is fitted independently inside each training fold.

No preprocessing statistic or tree is learned from the validation fold or final test set.


In [ ]:
gb_cv_raw = cross_validate(
    gb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(gb_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(gb_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(gb_cv_raw["score_time"][fold])
    fold_rows.append(row)

gb_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = gb_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

gb_cv_summary = pd.DataFrame(summary_rows)
display(gb_cv_folds)
display(gb_cv_summary)

gb_cv_folds.to_csv(MODEL_04_DIR / "cv_folds.csv", index=False)
gb_cv_summary.to_csv(MODEL_04_DIR / "cv_summary.csv", index=False)


## 4. Fit Frozen Model and Evaluate the Untouched Test Set

After CV, the exact same fixed configuration is fitted once on all training respondents.

The test set is evaluated once at probability threshold 0.5. No hyperparameter or threshold is changed afterward.


In [ ]:
gb_pipeline.fit(X_train, y_train)

y_test_prob_gb = gb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_gb = (y_test_prob_gb >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_gb, labels=[0, 1]).ravel()

test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_gb)),
    "precision": float(precision_score(y_test, y_test_pred_gb, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_gb)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_gb)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_gb)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_gb)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_gb)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "threshold": 0.5,
}
print(json.dumps(test_metrics, indent=2))

with open(MODEL_04_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)


## 5. Stratified Bootstrap Confidence Intervals

As with Models 01–03, uncertainty on fixed test predictions is estimated with 200 stratified bootstrap replicates. This procedure estimates uncertainty only; it does not tune the model.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251006

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_gb[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)
    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()

    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc","brier_score"]:
    values = bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

test_summary = pd.DataFrame(test_summary_rows)
display(test_summary)

bootstrap_df.to_csv(MODEL_04_DIR / "bootstrap_metrics.csv", index=False)
test_summary.to_csv(MODEL_04_DIR / "test_summary.csv", index=False)


## 6. Diagnostics and Feature Importance

ROC/PR curves and the fixed-threshold confusion matrix are saved.

Gradient Boosting impurity importance is exported as a diagnostic only. It is not causal evidence and is not used to reselect predictors.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_gb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_gb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Gradient Boosting (AUC={test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 04 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Gradient Boosting (AP={test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 04 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 04 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = gb_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = gb_pipeline.named_steps["model"].feature_importances_
importance_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "gini_importance": importances,
}).sort_values("gini_importance", ascending=False)

display(importance_table.head(20))
importance_table.to_csv(MODEL_04_DIR / "feature_importance_transformed.csv", index=False)


## 7. Freeze Rule

After this notebook completes successfully, Model 04 is frozen as:

> **100-stage Gradient Boosting classifier, log-loss objective, learning rate 0.1, depth-3 regression-tree weak learners, no resampling, threshold 0.5.**

No post-test change to stage count, learning rate, tree depth, threshold, preprocessing, or feature set is allowed. Any tuned Gradient Boosting configuration must be versioned as a separate experiment.
